In [1]:
import torch

In [2]:
print('hello')

hello


In [2]:
import os
import numpy as np
from glob import glob
import cv2
import SimpleITK as sitk


def save_npy(data, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.save(path, data)


def mr_norm(x, r=0.99):
    _x = np.sort(x.flatten())
    vmax = _x[int(len(_x) * r)]
    vmin = _x[0]
    x = np.clip(x, vmin, vmax)
    return (x - vmin) / (vmax - vmin + 1e-8)


def prepare_pmr(save_dir, ori_dir):
    site_names = os.listdir(ori_dir)
    site_names.sort()
    for i, site in enumerate(site_names):
        seg_paths = glob(os.path.join(ori_dir, site, '*mentation*'))
        seg_paths.sort()
        print('[INFO]', site, len(seg_paths))
        img_paths = [p[:-20] + '.nii.gz' for p in seg_paths]

        for j, seg_path in enumerate(seg_paths):
            itk_image = sitk.ReadImage(img_paths[j])
            itk_mask = sitk.ReadImage(seg_path)
            image = sitk.GetArrayFromImage(itk_image)
            mask = sitk.GetArrayFromImage(itk_mask)

            split = 'train' if j < int(len(seg_paths) * 0.8) else 'test'
            case_name = os.path.basename(img_paths[j])[:6]

            for k in range(image.shape[0]):
                base_name = f'slice_{k:03d}.npy'
                slice_image = mr_norm(image[k])
                slice_mask = mask[k] > 0
                if not slice_mask.max():
                    continue

                save_npy(slice_image,
                         f"{save_dir}/pmr/Site{i+1}/{split}/image/{case_name}/{base_name}")
                save_npy(slice_mask,
                         f"{save_dir}/pmr/Site{i+1}/{split}/mask/{case_name}/{base_name}")
        print(f"[DONE] {site}")


def prepare_fundus(save_dir, ori_dir):
    """
    Handles:
    /kaggle/input/lwr-dataset/New folder/Domain*/{train,test}/ROIs/{image,mask}
    Supports mixed file extensions (.png, .jpg, .bmp)
    """
    for domain_idx in range(1, 5):
        domain_dir = os.path.join(ori_dir, f"Domain{domain_idx}")
        if not os.path.exists(domain_dir):
            print(f"[WARN] Missing {domain_dir}")
            continue

        for split in ['train', 'test']:
            image_dir = os.path.join(domain_dir, split, 'ROIs', 'image')
            mask_dir = os.path.join(domain_dir, split, 'ROIs', 'mask')

            image_paths = sorted(glob(os.path.join(image_dir, '*')))
            if not image_paths:
                print(f"[WARN] No images found in {image_dir}")
                continue

            print(f"\n[INFO] Processing Domain{domain_idx}/{split} ({len(image_paths)} images)")

            for img_path in image_paths:
                base_name = os.path.splitext(os.path.basename(img_path))[0]

                # Find matching mask (handle .png/.jpg/.bmp)
                possible_masks = [
                    os.path.join(mask_dir, base_name + ext)
                    for ext in ['.png', '.jpg', '.bmp']
                ]
                mask_path = next((m for m in possible_masks if os.path.exists(m)), None)

                if not mask_path:
                    print(f"[WARN] No mask for {img_path}")
                    continue

                # Read image + mask
                img = cv2.imread(img_path)
                if img is None:
                    print(f"[WARN] Failed to read {img_path}")
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (384, 384), cv2.INTER_CUBIC)

                mask = cv2.imread(mask_path, 0)
                if mask is None:
                    print(f"[WARN] Failed to read {mask_path}")
                    continue
                mask = 2 - np.array(mask / 127, dtype='uint8')
                mask = cv2.resize(mask, (384, 384), cv2.INTER_NEAREST)

                # Save as npy
                save_npy(img, f"{save_dir}/fundus/Domain{domain_idx}/{split}/image/{base_name}")
                save_npy(mask, f"{save_dir}/fundus/Domain{domain_idx}/{split}/mask/{base_name}")

            print(f"[DONE] Domain{domain_idx}/{split}")


def prepare_polyp(save_dir, ori_dir):
    site_names = ['Kvasir', 'ETIS', 'CVC-ColonDB', 'CVC-ClinicDB']
    for split in ['train', 'test']:
        image_paths = [[], [], [], []]
        if split == 'train':
            for image_path in glob(os.path.join(ori_dir, 'TrainDataset', 'images', '*')):
                if image_path.split('/')[-1][0] == 'c':
                    image_paths[0].append(image_path)
                else:
                    image_paths[-1].append(image_path)
            image_paths[2] = [
                os.path.join(ori_dir, 'TestDataset', 'CVC-ColonDB', 'images', f'{i}.png')
                for i in list(range(1, 221)) + list(range(273, 380))
            ]
            image_paths[1] = [
                os.path.join(ori_dir, 'TestDataset', 'CVC-ColonDB', 'images', f'{i}.png')
                for i in range(1, 171)
            ]
        else:
            image_paths[0] = glob(os.path.join(ori_dir, 'TestDataset', 'Kvasir', 'images', '*'))
            image_paths[-1] = glob(os.path.join(ori_dir, 'TestDataset', 'CVC-ClinicDB', 'images', '*'))
            image_paths[2] = [
                os.path.join(ori_dir, 'TestDataset', 'CVC-ColonDB', 'images', f'{i}.png')
                for i in range(221, 273)
            ]
            image_paths[1] = [
                os.path.join(ori_dir, 'TestDataset', 'CVC-ColonDB', 'images', f'{i}.png')
                for i in range(171, 197)
            ]

        for i, site_name in enumerate(site_names):
            for image_path in image_paths[i]:
                mask_path = image_path.replace('images', 'masks')
                if not os.path.exists(mask_path):
                    print(f"[WARN] Missing mask {mask_path}")
                    continue

                img = cv2.imread(image_path)
                mask = cv2.imread(mask_path, 0)
                if img is None or mask is None:
                    continue

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (384, 384), cv2.INTER_CUBIC)
                mask = np.array(mask > 0, dtype='uint8')
                mask = cv2.resize(mask, (384, 384), cv2.INTER_NEAREST)

                file_name = os.path.splitext(os.path.basename(image_path))[0]
                save_npy(img, f"{save_dir}/polyp/{site_name}/{split}/image/{file_name}")
                save_npy(mask, f"{save_dir}/polyp/{site_name}/{split}/mask/{file_name}")
        print(f"[DONE] {split}")


if __name__ == '__main__':
    prepare_fundus(save_dir=r"C:\Users\adity\Downloads\dataset",
                   ori_dir=r"C:\Users\adity\Downloads\Fundus-doFE\Fundus")



[INFO] Processing Domain1/train (50 images)
[DONE] Domain1/train

[INFO] Processing Domain1/test (51 images)
[DONE] Domain1/test

[INFO] Processing Domain2/train (99 images)
[DONE] Domain2/train

[INFO] Processing Domain2/test (60 images)
[DONE] Domain2/test

[INFO] Processing Domain3/train (320 images)
[DONE] Domain3/train

[INFO] Processing Domain3/test (80 images)
[DONE] Domain3/test

[INFO] Processing Domain4/train (320 images)
[DONE] Domain4/train

[INFO] Processing Domain4/test (80 images)
[DONE] Domain4/test
